Pytorch Tutorial Notebook ver. SS2026 Die Erklärungen/Texte in diesem Notebook wurden (teilweise) mit Google Translate ins Deutsche übersetzt. Bitte entschuldige eventuelle Grammatik- oder Rechtschreibfehler.

-Jana van Leeuwen

# PyTorch

**PyTorch** ist die Bibliothek,
mit der heute fast alle neuronalen Sprachmodelle gebaut werden.

**Worum geht es?**

Text -> **Tensoren** -> **Embeddings** (RoBERTa) → ein kleines **neuronales Netz** ->
paarweise **Kanten-Scores** (mit `einsum`!) -> **Dependenzbaum**

**Die Form (`shape`) eines Tensors ist das Wichtigste auf dem Bildschirm.**

Fast jeder Bug in PyTorch ist in Wahrheit ein falsch geformter Tensor.


## 0. Setup

Wir brauchen für dieses Notebook nur `torch`. In Google Colab ist PyTorch schon
installiert. Lokal installiert ihr es mit `pip install torch`.

In [ ]:
import torch
print("PyTorch-Version:", torch.__version__)

# nur für Reproduzierbarkeit:
torch.manual_seed(42)



PyTorch-Version: 2.11.0+cpu


## 1. Was ist ein Tensor?

Ein **Tensor** ist einfach ein Behälter für Zahlen. PyTorch nennt alles "Tensor", damit es alles gleich behandeln kann:

| Mathe | Beispiel | Dimensionen (`ndim`) |
|---|---|---|
| **Skalar** | eine Zahl | 0 |
| **Vektor** | eine Zahlenliste | 1 |
| **Matrix** | eine Tabelle | 2 |
| **Tensor** | gestapelte Tabellen | 3+ |

In [ ]:
skalar = torch.tensor(42)
vektor = torch.tensor([1, 2, 3, 4, 5])
matrix = torch.tensor([[1, 2, 3],
                       [4, 5, 6]])
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])

for name, t in [("Skalar", skalar), ("Vektor", vektor),
                ("Matrix", matrix), ("3D-Tensor", tensor3d)]:
    print(f"{name:10s} shape={tuple(t.shape)}  ndim={t.ndim}")

Skalar     shape=()  ndim=0
Vektor     shape=(5,)  ndim=1
Matrix     shape=(2, 3)  ndim=2
3D-Tensor  shape=(2, 2, 2)  ndim=3


Lest die `shape` so:
- `(2, 3)` heißt **2 Zeilen, 3 Spalten**.
- `(2, 2, 2)` heißt **2 gestapelte 2×2-Tabellen**.

Merkt euch außerdem `dtype` (Datentyp). Das ist später wichtig: manche PyTorch-Funktionen
wollen `long` (Ganzzahlen, z.B. Token-IDs), andere `float` (Kommazahlen, z.B. Embeddings).

In [ ]:
ganzzahlen = torch.tensor([1, 2, 3]) #int
kommazahlen = torch.tensor([1.0, 2.0, 3.0]) #float
print("ganzzahlen :", ganzzahlen.dtype)
print("kommazahlen:", kommazahlen.dtype)

ganzzahlen : torch.int64
kommazahlen: torch.float32


## 2. Tensoren erzeugen und indizieren

Ein paar Wege, Tensoren zu erzeugen.

In [ ]:
print("arange(6):     ", torch.arange(6))
print("zeros(2,3):\n", torch.zeros(2, 3))
print("ones(2,3):\n",  torch.ones(2, 3))
print("randn(2,3):\n", torch.randn(2, 3)) # (mean 0, std 1)

arange(6):      tensor([0, 1, 2, 3, 4, 5])
zeros(2,3):
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones(2,3):
 tensor([[1., 1., 1.],
        [1., 1., 1.]])
randn(2,3):
 tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863]])


Indizieren und "slicen" funktioniert wie bei Python-Listen, nur in mehreren Dimensionen.
Der Doppelpunkt `:` bedeutet **"alles entlang dieser Achse"**.

In [ ]:
m = torch.arange(12).reshape(3, 4)
print("Matrix m:\n", m)
print("\nElement [0, 1]      :", m[0, 1])
print("Erste Zeile  m[0, :]:", m[0, :])
print("Erste Spalte m[:, 0]:", m[:, 0])
print("Teilmatrix m[0:2, 1:3]:\n", m[0:2, 1:3])


Matrix m:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

Element [0, 1]      : tensor(1)
Erste Zeile  m[0, :]: tensor([0, 1, 2, 3])
Erste Spalte m[:, 0]: tensor([0, 4, 8])
Teilmatrix m[0:2, 1:3]:
 tensor([[1, 2],
        [5, 6]])


## 3. Form-Gymnastik

Gleiche Zahlen, andere Form.

- `reshape` / `view`: ordnet die Zahlen in eine neue Form um.
- `transpose(a, b)`: vertauscht zwei Achsen (Zeilen <-> Spalten).
- `permute`: ordnet **mehrere** Achsen beliebig um.
- `unsqueeze(d)` / `squeeze()`: fügt eine Achse der Größe 1 hinzu / entfernt sie.

In [ ]:
x = torch.arange(6)
print("Original     ", tuple(x.shape), x.tolist())
print("reshape(2,3) \n", x.reshape(2, 3), "\n", tuple(x.reshape(2, 3).shape))
print("\nreshape(3,2) \n", x.reshape(3, 2), "\n",tuple(x.reshape(3, 2).shape))


Original      (6,) [0, 1, 2, 3, 4, 5]
reshape(2,3) 
 tensor([[0, 1, 2],
        [3, 4, 5]]) 
 (2, 3)

reshape(3,2) 
 tensor([[0, 1],
        [2, 3],
        [4, 5]]) 
 (3, 2)


In [ ]:
m = torch.arange(6).reshape(2, 3)
print("nm           ", tuple(m.shape))
print("m.transpose ", tuple(m.transpose(0, 1).shape)) # von (2,3) -> (3,2)
print("m.T         ", tuple(m.T.shape)) #2D

print("\nm.transpose \n", m.transpose(0, 1))
print("\nm.T \n", m.T)

nm            (2, 3)
m.transpose  (3, 2)
m.T          (3, 2)

m.transpose 
 tensor([[0, 3],
        [1, 4],
        [2, 5]])

m.T 
 tensor([[0, 3],
        [1, 4],
        [2, 5]])


In [ ]:
print("\nunsqueeze(0)", tuple(m.unsqueeze(0).shape)) # von (2,3) -> (1,2,3)
print("unsqueese 0 \n", m.unsqueeze(0))

print("unsqueeze(1)", tuple(m.unsqueeze(1).shape)) # von (2,3) -> (2,1,3)
print("unsqueese 1 \n", m.unsqueeze(1))


unsqueeze(0) (1, 2, 3)
unsqueese 0 
 tensor([[[0, 1, 2],
         [3, 4, 5]]])
unsqueeze(1) (2, 1, 3)
unsqueese 1 
 tensor([[[0, 1, 2]],

        [[3, 4, 5]]])


## 4. Rechnen mit Tensoren

**Elementweise** Operationen (`+`, `-`, `*`, `/`) arbeiten Zahl für Zahl. Achtung:
`*` ist **nicht** Matrixmultiplikation, sondern elementweise!

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[10., 20.], [30., 40.]])
print("a:", a)
print("\nb:", b)
print("\na + b:\n", a + b)
print("\na * b (elementweise!):\n", a * b)

a: tensor([[1., 2.],
        [3., 4.]])

b: tensor([[10., 20.],
        [30., 40.]])

a + b:
 tensor([[11., 22.],
        [33., 44.]])

a * b (elementweise!):
 tensor([[ 10.,  40.],
        [ 90., 160.]])


Für **Matrixmultiplikation** nehmt ihr `@` oder `torch.matmul`. Das ist die Operation
aus der 12. Vorlesung: eine Schicht eines Netzes rechnet (aus der vorherigen Schicht) $h = \sigma(W x + b)$ und das
$W x$ darin ist also eine Matrixmultiplikation.

In [ ]:
print("a @ b (Matrixmultiplikation):\n", a @ b)

a @ b (Matrixmultiplikation):
 tensor([[ 70., 100.],
        [150., 220.]])


**Broadcasting** ist PyTorchs Trick, kleinere und größere Tensoren automatisch
zusammenpassen zu lassen. Eine `(1,3)` und eine `(3,1)` ergeben zusammen eine `(3,3)`.
Praktisch... aber auch eine beliebte Quelle für stille Bugs.

In [ ]:
zeile  = torch.tensor([[1, 2, 3]])   # shape (1, 3)
spalte = torch.tensor([[10], [20], [30]])  # shape (3, 1)
print("Broadcasting-Ergebnis (3,3):\n", zeile * spalte)

Broadcasting-Ergebnis (3,3):
 tensor([[10, 20, 30],
        [20, 40, 60],
        [30, 60, 90]])


In [ ]:
zeile  = torch.tensor([[1, 2, 3], [4, 5, 6]])   # shape (2, 3)
spalte = torch.tensor([[10], [20]])  # shape (2, 1)
print("Broadcasting-Ergebnis (2,3):\n", zeile * spalte)

RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 0

In [ ]:
A = torch.randn(2, 3)
B = torch.randn(2, 3)
A @ B

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x3 and 2x3)

## 5. Sprache als Tensor

Jetzt wird es linguistisch. Ein Modell frisst keine Buchstaben, sondern Zahlen. Der Weg
von Text zu Tensor:

**Wörter -> IDs -> (evt. padding) -> Batch-Tensor -> Embeddings**

In [ ]:
saetze = [
    "Hans isst ein Käsebrot",
    "Maria gibt Hans ein Buch",
    "Der Hund schläft",
]

# Mini-Vokabular: jedes Wort bekommt eine Nummer
spezial = {"<PAD>": 0, "<UNK>": 1}
woerter = sorted({w.lower() for s in saetze for w in s.split()})
vocab = {w: i + len(spezial) for i, w in enumerate(woerter)}
print("Vokabular:", vocab)

# Wörter -> IDs
ids = [[vocab.get(w.lower()) for w in s.split()] for s in saetze]
for s, i in zip(saetze, ids):
    print(f"{s!r:35s} -> {i}")


Vokabular: {'buch': 2, 'der': 3, 'ein': 4, 'gibt': 5, 'hans': 6, 'hund': 7, 'isst': 8, 'käsebrot': 9, 'maria': 10, 'schläft': 11}
'Hans isst ein Käsebrot'            -> [6, 8, 4, 9]
'Maria gibt Hans ein Buch'          -> [10, 5, 6, 4, 2]
'Der Hund schläft'                  -> [3, 7, 11]


Das Vokabular ist extrem limitiert!

In [ ]:

unksaetze = [
    "Hans isst ein Käsebrot mit Butter",
    "Maria gab Hans ein Buch",
    "Der Hund schläft immer",
]


# Wörter -> IDs
ids = [[vocab.get(w.lower()) for w in s.split()] for s in unksaetze]
for s, i in zip(unksaetze, ids):
    print(f"{s!r:35s} -> {i}")

'Hans isst ein Käsebrot mit Butter' -> [6, 8, 4, 9, None, None]
'Maria gab Hans ein Buch'           -> [10, None, 6, 4, 2]
'Der Hund schläft immer'            -> [3, 7, 11, None]


In [ ]:
# Mini-Vokabular: jedes Wort bekommt eine Nummer
# Jetzt mit "Special Token" für unknown words
spezial = {"<UNK>": 0}
woerter = sorted({w.lower() for s in saetze for w in s.split()})
vocab = {**spezial, **{w: i + len(spezial) for i, w in enumerate(woerter)}}
print("Vokabular:", vocab)

# Wörter -> IDs
ids = [[vocab.get(w.lower(), vocab["<UNK>"]) for w in s.split()] for s in unksaetze]
for s, i in zip(unksaetze, ids):
    print(f"{s!r:35s} -> {i}")

Vokabular: {'<UNK>': 0, 'buch': 1, 'der': 2, 'ein': 3, 'gibt': 4, 'hans': 5, 'hund': 6, 'isst': 7, 'käsebrot': 8, 'maria': 9, 'schläft': 10}
'Hans isst ein Käsebrot mit Butter' -> [5, 7, 3, 8, 0, 0]
'Maria gab Hans ein Buch'           -> [9, 0, 5, 3, 1]
'Der Hund schläft immer'            -> [2, 6, 10, 0]


Die Sätze sind **unterschiedlich lang**, ein Tensor muss aber rechteckig sein. Lösung:
**Padding**
Wir füllen kürzere Sätze mit dem `<PAD>`-Token (ID 0) auf, bis alle gleich
lang sind. Heraus kommt die Form `(batch_size, seqlen)`


In [ ]:
# Mini-Vokabular: jedes Wort bekommt eine Nummer
# Jetzt mit "Special Token" für unknown words
# Und mit Padding-Tokens
spezial = {"<PAD>":0, "<UNK>": 1}
woerter = sorted({w.lower() for s in saetze for w in s.split()})
vocab = {**spezial, **{w: i + len(spezial) for i, w in enumerate(woerter)}}
print("Vokabular:", vocab)

# Wörter -> IDs
ids = [[vocab.get(w.lower(), vocab["<UNK>"]) for w in s.split()] for s in unksaetze]
for s, i in zip(unksaetze, ids):
    print(f"{s!r:35s} -> {i}")

Vokabular: {'<PAD>': 0, '<UNK>': 1, 'buch': 2, 'der': 3, 'ein': 4, 'gibt': 5, 'hans': 6, 'hund': 7, 'isst': 8, 'käsebrot': 9, 'maria': 10, 'schläft': 11}
'Hans isst ein Käsebrot mit Butter' -> [6, 8, 4, 9, 1, 1]
'Maria gab Hans ein Buch'           -> [10, 1, 6, 4, 2]
'Der Hund schläft immer'            -> [3, 7, 11, 1]


In [ ]:
maxlen = max(len(i) for i in ids)
ids_padded = [i + [vocab["<PAD>"]] * (maxlen - len(i)) for i in ids]

x = torch.tensor(ids_padded, dtype=torch.long)   # long, weil es IDs sind!
print("Batch-Tensor x:\n", x)
print("\nshape (batch_size, seqlen):", tuple(x.shape))

Batch-Tensor x:
 tensor([[ 6,  8,  4,  9,  1,  1],
        [10,  1,  6,  4,  2,  0],
        [ 3,  7, 11,  1,  0,  0]])

shape (batch_size, seqlen): (3, 6)


**Attention-Mask bauen** "0" hat nämlich gar keine "Bedeutung"... Welche Position ist echt, welche ist nur Füllmaterial?

In [ ]:
attention_mask = (x != vocab["<PAD>"]).long()
print(attention_mask)

tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 0, 0]])


Eine ID wie `7` bedeutet für sich genommen nichts. Deshalb bilden wir jede ID auf einen
**Vektor** ab mit **Word Embedding**. Diese Vektoren werden beim Training gelernt, sodass
ähnliche Wörter ähnliche Vektoren bekommen. In PyTorch macht das `nn.Embedding`.

In [ ]:
import torch.nn as nn

embedding_dim = 16
embedding = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embedding_dim)

embeddings = embedding(x)   # x ist (bs, seqlen) -> Ausgabe (bs, seqlen, embedding_dim)
print("shape (batch, seqlen, embedding_dim):", tuple(embeddings.shape))

shape (batch, seqlen, embedding_dim): (3, 6, 16)


### Tip:
Genau das macht **XLM-RoBERTa** im Parser (nur viel besser und mit Kontext). Statt eines frisch initialisierten 16-dimensionalen Embeddings bekommt ihr dort einen Tensor der Form **`(bs, seqlen, 768)`**: für jedes Token in jedem Satz einen 768-dimensionalen, *vortrainierten* Vektor. Die Form ist dieselbe Idee, die wir hier gerade gebaut haben.

In [ ]:
# nur mit Internet:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
print(tok.tokenize("Katze"))
print(tok.tokenize("Katzenliebhaberin")) # mehrere Subword-Stücke

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

['▁Kat', 'ze']
['▁Ka', 'tzen', 'lieb', 'haber', 'in']


## 6. Ein **winziges** neuronales Netz

Aus der Vorlesung: Ein neuronales Netz ist eine Kette von Schichten der Form
$h = \sigma(W x + b)$. In PyTorch ist

- eine **lineare Schicht** = `nn.Linear` (das $W x + b$),
- die **Nichtlinearität** $\sigma$ = z.B. `nn.ReLU`,
- das **ganze Netz** = eine Klasse, die von `nn.Module` erbt.

Das ganze Modell steckt in zwei Methoden:
- `__init__`: hier listen wir die Schichten auf.
- `forward`: hier beschreiben wir, wie die Daten durchfließen.

In [ ]:
import torch.nn.functional as F

class MiniMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.schicht1 = nn.Linear(input_dim, hidden_dim)   # W1 x + b1
        self.schicht2 = nn.Linear(hidden_dim, num_classes) # W2 h + b2

    def forward(self, x):
        h = self.schicht1(x)
        h = F.relu(h) # Nichtlinearität
        out = self.schicht2(h) # rohe Scores (Logits)
        return out

modell = MiniMLP(input_dim=16, hidden_dim=32, num_classes=3)
print(modell)

# Ein einzelner Forward-Pass
beispiel = torch.randn(5, 16) # 5 "Wörter" mit je 16 Features
logits = modell(beispiel)
print("\nInput-shape :", tuple(beispiel.shape))
print("Output-shape:", tuple(logits.shape), "= (5 Wörter, 3 Klassen)")

MiniMLP(
  (schicht1): Linear(in_features=16, out_features=32, bias=True)
  (schicht2): Linear(in_features=32, out_features=3, bias=True)
)

Input-shape : (5, 16)
Output-shape: (5, 3) = (5 Wörter, 3 Klassen)


In [ ]:
d = 5
U1 = nn.Parameter(torch.empty(d, d)) # leer = uninitialisiert!
nn.init.xavier_uniform_(U1) # sinnvolle Startwerte
u2 = nn.Parameter(torch.zeros(d)) # Bias-Vektoren startet man oft mit 0
print("U1 shape:", tuple(U1.shape), " u2 shape:", tuple(u2.shape))

U1 shape: (5, 5)  u2 shape: (5,)


## 7. `einsum`: die schönste Funktion in PyTorch

Aus der Vorlesung: *"Einsum ist die beste Pytorch-Funktion!"*

`einsum` ("Einstein-Summenkonvention") ist eine kompakte Schreibweise für
Tensor-Operationen. Die Regel:

1. Benennt die Achsen jedes Eingabe-Tensors mit Buchstaben.
2. Schreibt nach dem `->`, welche Achsen das Ergebnis haben soll.
3. Über jeden Buchstaben, der **links** vorkommt, aber **rechts fehlt**, wird *summiert*.

Ein paar Beispiele:

In [ ]:
m = torch.arange(6).reshape(2, 3)
v = torch.arange(3)

print("Matrix m: \n", m)

# Transponieren: Achsen i und j vertauschen
print("\n transpose 'ij->ji':\n", torch.einsum("ij->ji", m))

# Skalarprodukt zweier Vektoren: über i summieren
a, b = torch.arange(3), torch.arange(3, 6)
print("\na: ", a)
print("\nb: ", b)
print("\ndot 'i,i->':", torch.einsum("i,i->", a, b).item())

a= torch.arange(3)
b = torch.arange(3)
print("\na: ", a)
print("\nb: ", b)
print("\ndot 'i,j->':", torch.einsum("i,j->", a, b).item())

# Matrix x Vektor: über k summieren
print("\nmatvec 'ik,k->i':", torch.einsum("ik,k->i", m, v))

# Matrixmultiplikation: über k summieren
A = torch.arange(6).reshape(2, 3)
B = torch.arange(15).reshape(3, 5)
print("\nmatmul 'ik,kj->ij' -> shape", tuple(torch.einsum("ik,kj->ij", A, B).shape))

Matrix m: 
 tensor([[0, 1, 2],
        [3, 4, 5]])

 transpose 'ij->ji':
 tensor([[0, 3],
        [1, 4],
        [2, 5]])

a:  tensor([0, 1, 2])

b:  tensor([3, 4, 5])

dot 'i,i->': 14

a:  tensor([0, 1, 2])

b:  tensor([0, 1, 2])

dot 'i,j->': 9

matvec 'ik,k->i': tensor([ 5, 14])

matmul 'ik,kj->ij' -> shape (2, 5)


Auch wichtig: **Batch-Dimension** `b`: damit rechnen
wir für *alle Sätze im Batch gleichzeitig*, ohne eine Schleife zu schreiben.

In [ ]:
bs = 4
seqlen = 6
dim = 8
H1 = torch.randn(bs, seqlen, dim)
H2 = torch.randn(bs, seqlen, dim)

print("H1:", H1)
print("\nH2:", H2)

# Batch-Matrixmultiplikation: 'b' bleibt erhalten, über 'k' wird summiert
batch_mm = torch.einsum("bik,bkj->bij", torch.randn(bs, seqlen, dim),
                                          torch.randn(bs, dim, seqlen))
print("Batch-Matmul-Ergebnis:", tuple(batch_mm.shape))
print("\nbatch_mm:", batch_mm)

H1: tensor([[[-2.2118e+00, -5.7546e-01,  4.3194e-01,  8.3380e-01, -1.2662e-01,
          -4.5185e-02,  1.8064e-01,  5.3500e-01],
         [-6.5526e-02,  6.1104e-01,  6.6490e-01, -2.4996e-01,  2.1618e-01,
          -6.1230e-01, -2.5462e-01,  9.9259e-01],
         [-1.0256e+00,  1.7627e+00, -2.8158e+00, -2.8375e-01,  1.9666e+00,
           4.4047e-01, -5.9434e-01,  7.2064e-01],
         [ 3.1439e-01, -6.6913e-02, -6.5468e-01,  1.0335e+00, -3.1327e-01,
          -1.8329e-01, -1.7837e-01,  2.0791e+00],
         [ 9.8329e-01,  2.7253e-01,  7.2023e-01,  8.7691e-01, -8.5683e-01,
          -1.4692e+00,  9.2607e-01, -5.5944e-01],
         [-5.7850e-01, -7.2574e-01,  1.8645e+00, -8.4391e-01, -1.0606e+00,
           1.0454e+00,  1.1547e+00,  3.2125e-01]],

        [[-1.3895e-01, -5.1991e-01, -4.2977e-01, -9.3303e-01,  3.4458e-02,
           5.0891e-01,  6.4701e-01, -5.0199e-01],
         [ 1.8565e-02, -1.6756e+00, -1.9437e+00,  9.8655e-02,  2.5731e+00,
          -7.0517e-01, -1.1590e+00, -2.7696e

### Die "language-y" Anwendung

Beim graphbasierten Parsing wollen wir für **jedes Paar von Tokens** $(i, k)$ einen
Score, der sagt wie plausibel eine Kante von Wort $i$ zu Wort $k$ ist.

Das einfachste Scoring-Modell aus der Vorlesung ist das Skalarprodukt der beiden
Token-Vektoren: $s_{i\to k} = h_i \cdot h_k$. Mit `einsum` über die ganze Batch:

In [ ]:
# H: für jedes Token einen Vektor, so etwas liefert RoBERTa (hier simuliert)
bs, seqlen, dim = 4, 6, 8
H = torch.randn(bs, seqlen, dim)

# s[b,i,k] = sum_d H[b,i,d] * H[b,k,d]
scores = torch.einsum("bid,bkd->bik", H, H)

print("H      shape:", tuple(H.shape), "= (Batch, Seqlen, Dim)")
print("scores shape:", tuple(scores.shape), "= (Batch, Seqlen, Seqlen)")
print("\nscores[b, i, k] = Score der Kante von Token i zu Token k im Satz b")

H      shape: (4, 6, 8) = (Batch, Seqlen, Dim)
scores shape: (4, 6, 6) = (Batch, Seqlen, Seqlen)

scores[b, i, k] = Score der Kante von Token i zu Token k im Satz b


## 8. Die Trainingsschleife

Wie lernt ein Netz? Immer dasselbe Verfahren, viele tausend Mal wiederholt:

1. **`zero_grad`**: die Notizen vom letzten Mal löschen
2. **`forward`**: Daten durchschicken, Vorhersage holen.
3. **`loss`**: eine Zahl ausrechnen, *wie falsch* wir lagen
4. **`backward` + `step`**: ausrechnen, in welche Richtung jedes Gewicht muss, und einen
   kleinen Schritt dorthin machen (gradient descent)

Zurücksetzen -> Vorhersagen -> Messen -> Anpassen.

Für Klassifikation ist die Verlustfunktion die **negative log-likelihood**, in PyTorch
`nn.CrossEntropyLoss`. Hier ein winziges, **lauffähiges** Beispiel. Als Aufgabe nehmen wir **zwei konzentrische Ringe**: ein
innerer Ring (Klasse 0) und ein äußerer Ring (Klasse 1). Diese Punkte sind **nicht** mit
einer geraden Linie trennbar. Deshalb brauchen wir die **ReLU-Nichtlinearität**.

In [ ]:
import math
# Zwei Ringe (nicht linear trennbar)
torch.manual_seed(0)
n = 150
r0 = torch.rand(n) * 1.6;        a0 = torch.rand(n) * 2 * math.pi # innerer Ring
r1 = 1.3 + torch.rand(n) * 1.5;  a1 = torch.rand(n) * 2 * math.pi # äußerer Ring
X0 = torch.stack([r0 * torch.cos(a0), r0 * torch.sin(a0)], dim=1)
X1 = torch.stack([r1 * torch.cos(a1), r1 * torch.sin(a1)], dim=1)
X = torch.cat([X0, X1])
y = torch.cat([torch.zeros(n), torch.ones(n)]).long()
perm = torch.randperm(len(X)); X, y = X[perm], y[perm]
print("X:", tuple(X.shape), " y:", tuple(y.shape), " Klassen:", y.unique().tolist())

X: (300, 2)  y: (300,)  Klassen: [0, 1]


In [ ]:
# Modell, Loss, Optimierer
modell = MiniMLP(input_dim=2, hidden_dim=32, num_classes=2)
loss_fn = nn.CrossEntropyLoss()
optimierer = torch.optim.Adam(modell.parameters(), lr=0.01)

# 150 Epochen
modell.train()
for epoche in range(150):
    optimierer.zero_grad() # 1. zurücksetzen
    logits = modell(X) # 2. forward
    loss = loss_fn(logits, y) # 3. messen
    loss.backward() # 4a. Gradienten ausrechnen
    optimierer.step() # 4b. Schritt machen

    if epoche % 25 == 0 or epoche == 149:
        genauigkeit = (logits.argmax(dim=1) == y).float().mean()
        print(f"Epoche {epoche:3d} | Loss {loss.item():.3f} | Genauigkeit {genauigkeit:.2%}")

Epoche   0 | Loss 0.664 | Genauigkeit 47.00%
Epoche  25 | Loss 0.260 | Genauigkeit 91.67%
Epoche  50 | Loss 0.167 | Genauigkeit 92.33%
Epoche  75 | Loss 0.147 | Genauigkeit 92.33%
Epoche 100 | Loss 0.139 | Genauigkeit 92.33%
Epoche 125 | Loss 0.135 | Genauigkeit 92.67%
Epoche 149 | Loss 0.133 | Genauigkeit 92.67%


Loss fällt, Genauigkeit steigt, das Netz lernt. Dass es bei ~93 % stehen
bleibt und nicht 100 % erreicht, ist hier völlig normal: die beiden Ringe überlappen,
es gibt also gar keine perfekte Trennung. Echte Daten sind fast immer so.


In [ ]:
# Evaluation: kein Gradient nötig
modell.eval()
with torch.no_grad():
    pred = modell(X).argmax(dim=1)
    print("Genauigkeit (eval):", (pred == y).float().mean().item())

Genauigkeit (eval): 0.9266666769981384


## 9.Tipps

Spart euch viel Frust! (beim Übungsblatt und überhaupt)

### 9.1 GPU vs. CPU
(Fast) Alles in diesem Notebook läuft in Sekunden auf der CPU. Sobald **RoBERTa** im Spiel ist
(im Übungsblatt), braucht ihr eine **GPU**, sonst dauert das Training ewig. In Colab:
*Laufzeit -> Laufzeittyp ändern -> GPU*. Im Code:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Wir rechnen auf:", device)

# WICHTIG: Modell UND Daten müssen auf demselben Gerät liegen!
modell = modell.to(device)
X = X.to(device)
# ansonsten: "Expected all tensors to be on the same device"

Wir rechnen auf: cpu


### 9.2 Shapes sind euer bester Freund
Wenn etwas nicht funktioniert, ist es zu 90 % eine falsche Form. Zwei Gewohnheiten:
- Sei großzügig mit: `print(x.shape)`
- **Behauptet Shapes** mit `assert` (so merkt ihr Fehler sofort an der richtigen Stelle.)

In [ ]:
def score_kanten(H):
    bs, seqlen, dim = H.shape
    scores = torch.einsum("bid,bkd->ik", H, H)
    assert scores.shape == (bs, seqlen, seqlen), f"falsche Form: {scores.shape}"
    return scores

print("Test bestanden, shape:", tuple(score_kanten(torch.randn(2, 5, 8)).shape))

AssertionError: falsche Form: torch.Size([5, 5])

### 9.3 Datentypen (`dtype`)
- `nn.Embedding` und `CrossEntropyLoss`-**Labels** wollen `long` (Ganzzahlen).
- Embeddings/Features sind `float32`.
- Typische Fehlermeldung *"expected scalar type Long but found Float"* ->
  irgendwo `dtype` verwechselt. Umwandeln mit `.long()` bzw. `.float()`.

### 9.4 Weights & Biases (W&B) (fürs Übungsblatt wichtig)
W&B zeichnet eure **Lernkurven** auf (Loss & Genauigkeit pro Epoche). Legt euch vorab
einen kostenlosen Account an und loggt euch einmal ein. Das Grundmuster (hier nur zum Verständnis):

```python
import wandb
wandb.login() # einmalig, fragt nach API-Key
wandb.init(project="mein-parser", config={"lr": 5e-5, "batch_size": 1})
for epoche in range(num_epochs):
    # trainieren weeeeee
    wandb.log({"train/loss": loss, "dev/uas": uas}) # landet als Kurve im Dashboard
```

### 9.5 Lernkurven lesen (aus der Vorlesung)
- **Trainings-Loss fällt Richtung 0** -> das Modell lernt korrekt aus den Trainingsdaten.
- **Dev-Genauigkeit sinkt, während der Trainings-Loss weiter fällt** -> **Overfitting**.
  Das Modell merkt sich die Trainingsdaten auswendig, statt zu verallgemeinern.
- Trainiert immer **lange genug**, bis sich Loss/Genauigkeit stabilisieren.



## Mini-Zusammenfassung :)


|  |  |  |
|---|---|---|
| Text -> Zahlen | `nn.Embedding` (Abschnitt 5) | XLM-RoBERTa -> `(bs, seqlen, 768)` |
| Schichten / MLP | `nn.Module`, `nn.Linear`, `ReLU` (6) | `MLP_head`, `MLP_dep` |
| eigene Parameter | `nn.Parameter` + Init (6) | `U1`, `u2` |
| paarweise Scores | `einsum` -> `(bs, seqlen, seqlen)` (7) | Kanten-Scores `scores[b,i,k]` |
| Lernen | Vierertakt-Schleife (8) | schon für euch geschrieben |
| Beobachten | W&B, Lernkurven (9) | W&B-Screenshots abgeben |


*Auch hilfreich: `Einsum.ipynb` mit mehr Beispielen, falls ihr `einsum` noch
weiter vertiefen wollt.*